# Sprint 1 — pCTR Model Training
### AI-Powered YouTube Title Optimizer — Random Forest + XGBoost
Reference: `SOURCE_OF_TRUTH.md` §16 | `PLAN_IMPLEMENTATION.md` §1.5

**Prerequisite:** run `data_exploration.ipynb` first and have `labeled_titles.csv` ready to upload.

**Exit criteria (PLAN §1.6):** F1 > 0.70 on validation set (target F1 > 0.75 on test), feature importance plot with VADER/has_number/title_length as top signals, `pctr_model.pkl` saved.

In [ ]:
!pip -q install scikit-learn xgboost joblib pandas matplotlib scipy

In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("labeled_titles.csv")

if IN_COLAB and not DATA_PATH.exists():
    from google.colab import files
    print("Upload labeled_titles.csv (produced by data_exploration.ipynb):")
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).rename(DATA_PATH) if fname != str(DATA_PATH) else None

print("Data path:", DATA_PATH, "exists:", DATA_PATH.exists())

## 1. Load data and prepare features/target

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} cols")
df.head()

In [ ]:
FEATURE_COLS = [
    "title_length_chars", "title_word_count", "has_number", "number_value",
    "vader_compound", "vader_positive", "has_question", "has_exclamation",
    "all_caps_ratio", "has_emoji", "category_id",
]
CATEGORICAL_COLS = ["first_word_type", "channel_subscriber_tier"]

def prepare_xy(df):
    df = df.copy()
    encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        df[col + "_enc"] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
    feature_cols = FEATURE_COLS + [c + "_enc" for c in CATEGORICAL_COLS]
    X = df[feature_cols].astype(float)
    y = df["label"].astype(int)
    return X, y, encoders, feature_cols

X, y, encoders, feature_cols = prepare_xy(df)
print(X.shape, y.value_counts().to_dict())

## 2. Split — 70% train / 15% val / 15% test (SOT §16.2)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f"train={len(X_train)} val={len(X_val)} test={len(X_test)}")

## 3. Train Random Forest + XGBoost with RandomizedSearchCV

In [ ]:
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

def train_random_forest(X_train, y_train):
    param_dist = {
        "n_estimators": randint(100, 500),
        "max_depth": randint(10, 30),
        "min_samples_split": randint(2, 10),
    }
    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_dist, n_iter=15, cv=3, scoring="f1", random_state=42, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    print("RF best params:", search.best_params_)
    return search.best_estimator_

def train_xgboost(X_train, y_train):
    param_dist = {
        "learning_rate": uniform(0.01, 0.09),
        "n_estimators": randint(100, 500),
        "max_depth": randint(3, 8),
        "reg_alpha": uniform(0, 1),
        "reg_lambda": uniform(0, 1),
    }
    search = RandomizedSearchCV(
        XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
        param_dist, n_iter=15, cv=3, scoring="f1", random_state=42, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    print("XGB best params:", search.best_params_)
    return search.best_estimator_

rf = train_random_forest(X_train, y_train)
xgb = train_xgboost(X_train, y_train)

## 4. Evaluate on validation set (SOT §16.3)

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

def evaluate(model, X, y, name):
    preds = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    metrics = {
        "f1": f1_score(y, preds),
        "roc_auc": roc_auc_score(y, proba),
        "precision": precision_score(y, preds),
        "recall": recall_score(y, preds),
    }
    print(f"[{name}] " + " ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    return metrics

rf_metrics = evaluate(rf, X_val, y_val, "RandomForest-val")
xgb_metrics = evaluate(xgb, X_val, y_val, "XGBoost-val")

best_name, best_model, best_metrics = (
    ("xgboost", xgb, xgb_metrics) if xgb_metrics["f1"] >= rf_metrics["f1"]
    else ("random_forest", rf, rf_metrics)
)
print(f"\nBest model: {best_name}")

## 5. Final evaluation on held-out test set

In [ ]:
test_metrics = evaluate(best_model, X_test, y_test, f"{best_name}-test")
print(f"\nExit criteria (F1 > 0.70 val): {'PASS' if best_metrics['f1'] > 0.70 else 'FAIL'}")
print(f"Target (F1 > 0.75 test): {'PASS' if test_metrics['f1'] > 0.75 else 'FAIL'}")

## 6. Feature importance (top 10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

importances = best_model.feature_importances_
order = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(8, 5))
plt.barh([feature_cols[i] for i in order][::-1], importances[order][::-1])
plt.xlabel("Importance")
plt.title(f"Top 10 Feature Importances ({best_name})")
plt.tight_layout()
plt.savefig(MODELS_DIR / "feature_importance.png")
plt.show()

## 7. Save model + training report

In [ ]:
import json
import joblib

joblib.dump({
    "model": best_model,
    "model_name": best_name,
    "feature_cols": feature_cols,
    "categorical_encoders": encoders,
}, MODELS_DIR / "pctr_model.pkl")

with open(MODELS_DIR / "training_report.json", "w") as f:
    json.dump({
        "best_model": best_name,
        "val_metrics": {k: float(v) for k, v in best_metrics.items()},
        "test_metrics": {k: float(v) for k, v in test_metrics.items()},
        "rf_val_metrics": {k: float(v) for k, v in rf_metrics.items()},
        "xgb_val_metrics": {k: float(v) for k, v in xgb_metrics.items()},
    }, f, indent=2)

print(f"Saved -> {MODELS_DIR / 'pctr_model.pkl'}")
print(f"Saved -> {MODELS_DIR / 'training_report.json'}")

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(MODELS_DIR / "pctr_model.pkl"))
    files.download(str(MODELS_DIR / "training_report.json"))
    files.download(str(MODELS_DIR / "feature_importance.png"))

**Next:** copy the three downloaded files into `system/models/` in the repo, then check off Sprint 1.5/1.6 in `PLAN_IMPLEMENTATION.md`.